Evaluating whether newly proposed features earn a place in an existing model.

The worked example is the [UCI Taiwanese Bankruptcy dataset](https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction): 6,819 companies, 95 financial ratios, a 3.2% bankruptcy rate. The incumbent model uses the profitability, leverage and growth ratios; the **cash-flow family** is the candidate set under evaluation. Two controls are planted among the candidates — `cand_dup_roa_c` (a near-copy of an incumbent) and `cand_noise` (pure noise) — so the screens can be checked against a known answer.

| Section | Covers |
| --- | --- |
| [1. The five stages](#1) | What each stage takes and returns, called directly |
| [2. One run](#2) | The same chain through `Pipeline`, and the artifacts it writes |
| [3. The discovery loop](#6) | Propose, verify, keep or try again |
| [Configuration](#7) | Parallelism, gates, tuning |

**Prerequisite:** `python data/bankruptcy/prepare.py` (once), which writes `data/bankruptcy/modeling.parquet`.

> **If a cell fails with an error that does not match the source you are reading** — an unknown config key, an argument the signature does not have — the kernel is holding an older copy of `mllite`. Use **Kernel → Restart & Run All**. The canary in the setup cell catches this at the top rather than midway through, but it cannot rescue a kernel that has already imported the module.

In [3]:
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)   # config paths are relative to the project root, as from the CLI

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 60)

# mllite is edited alongside this notebook, so keep the kernel in step.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import inspect

import mllite
from mllite import Pipeline, build_dataset, load_config, prepare_dataset
from mllite.stages.analysis import run_analysis
from mllite.stages.data_quality import run_data_quality
from mllite.stages.feature_selection import run_feature_selection
from mllite.stages.modeling import build_variants, run_modeling
from mllite.stages.verdict import run_verdict

# Staleness canary. A kernel that imported mllite before the library was last
# edited fails later in ways whose tracebacks make no sense. Loading the shipped
# configs against the imported dataclasses catches any drift between the two,
# including config keys added after this kernel started - which is exactly the
# kind of change `%autoreload` cannot apply, since it patches methods but does
# not rebuild dataclass fields.
for _config in sorted(Path('configs').glob('*.yaml')):
    try:
        load_config(_config)
    except Exception as exc:
        raise RuntimeError(
            f'This kernel holds an older copy of mllite: {_config} does not load 'f'({exc}). Kernel -> Restart & Run All.') from None

print(f"mllite {mllite.__version__} from {Path(mllite.__file__).parent}")
for module in ('numpy', 'pandas', 'xgboost', 'shap'):
    print(f'  {module:<8} {__import__(module).__version__}')

mllite 0.1.0 from /Users/mingxuanliu/Library/CloudStorage/GoogleDrive-mingxuan99michelle@gmail.com/My Drive/Projs/mllite/src/mllite
  numpy    2.0.0
  pandas   2.3.2
  xgboost  1.7.6
  shap     0.49.1


<a id='5'></a>
### Data that is already split

When train, valid and test are prepared upstream, nothing should re-split them. There are four input shapes and they all produce the same `Dataset`:

| Your data | How |
| --- | --- |
| One table, let the pipeline split it | `data.path` + `data.split` |
| One table carrying a split column | `data.split.mode: column` |
| Already split, separate files | `data.paths: {train: …, valid: …, test: …}` |
| Already split, frames in memory | `run(frames={'train': df, …})` |

With `data.paths` or `frames=`, the `data.split` block is ignored entirely and the rows are taken exactly as given.

In [26]:
cfg_presplit = load_config('configs/bankruptcy.yaml')
cfg_presplit.run.name = 'presplit_demo'
cfg_presplit.run.log_level = 'ERROR'
cfg_presplit.model.variants = ['base', 'base_plus_new']
cfg_presplit.analysis.shap.enabled = False

# Stand in for sets prepared upstream; however they were built, the row
# assignment below is preserved exactly.
bank = pd.read_parquet(cfg_presplit.data.path)
draw = np.random.default_rng(0).random(len(bank))
prepared = {'train': bank[draw >= 0.4],
            'valid': bank[(draw >= 0.2) & (draw < 0.4)],
            'test':  bank[draw < 0.2]}

# presplit = Pipeline(cfg_presplit).run(frames=prepared)

# for split, given in prepared.items():
#     used = presplit.dataset.split(split)
#     print(f'{split:<6} given {len(given):>5,} -> used {len(used):>5,}   event rate {used[cfg_presplit.data.target].mean():.2%}')

<a id='1'></a>
## 1. The five stages

`Pipeline` is a thin sequencer over five stage functions. Calling them directly shows what actually moves between them: **one `Dataset` flows down the chain**, each stage returns its own result object, and those result objects are terminal — they are written to disk and read by you, never by the next stage.

| Stage | Call | Returns |
| --- | --- | --- |
| 0 | `prepare_dataset(frame, cfg)` | `Dataset` |
| 1 | `run_data_quality(dataset, cfg)` | `DataQualityResult` |
| 2 | `run_feature_selection(dataset, cfg)` | `FeatureSelectionResult` |
| 3 | `run_modeling(dataset, cfg, model_dir)` | `list[ModelResult]`, tuning log |
| 4 | `run_analysis(dataset, cfg, models)` | `AnalysisResult` |
| 5 | `run_verdict(candidates, cfg, ...)` | verdict table, `BatchVerdict` |

### Stage 0 — build the Dataset

Loads the table, resolves which columns are incumbent and which are candidates, converts sentinel codes to `NaN`, and splits.

In [37]:
cfg = load_config('configs/bankruptcy.yaml')
cfg.run.log_level = 'WARNING'   # quiet for the notebook

frame = pd.read_parquet(cfg.data.path)
dataset = prepare_dataset(frame, cfg)

print(f'{cfg.data.path}  ->  {frame.shape[0]:,} rows x {frame.shape[1]} columns')
print(dataset.describe())
print(cfg.features.new)

data/bankruptcy/modeling.parquet  ->  6,819 rows x 99 columns
{'rows_per_split': {'train': 4091, 'valid': 1364, 'test': 1364}, 'n_base_features': 84, 'n_new_features': 13, 'target': 'bankrupt'}
['cash_flow_rate', 'cash_flow_per_share', 'cash_reinvestment_pct', 'cash_total_assets', 'cash_current_liability', 'cash_turnover_rate', 'cash_flow_to_sales', 'cash_flow_to_total_assets', 'cash_flow_to_liability', 'cfo_to_assets', 'cash_flow_to_equity', 'cand_dup_roa_c', 'cand_noise']


### Stage 1 — data quality

`data_quality.enabled` turns the stage on; `drop_failed` decides whether failing features are removed or only reported. Two families of check run:

- **Per-feature quality** — missing rate and cardinality.
- **Distribution consistency** — is a variable shaped the same way in valid and test as it is in train? Measured as PSI against `distribution_reference`, with NaN as its own bucket so a variable that is 2% null in train and 40% null in test registers as shifted rather than being quietly binned away.

| PSI | Reading |
| --- | --- |
| < 0.10 | no meaningful shift |
| 0.10 – 0.25 | moderate, worth a look |
| >= 0.25 | not the same variable in the two samples |

Period stability across a `by_col` column is a separate check and is still the unported AIME_DataStability piece.

In [39]:
dq = run_data_quality(dataset, cfg)

print('thresholds applied:')
for name, value in dq.thresholds.items():
    print(f'  {name:<18} {value:g}')
print(f"\n{dq.summary()['n_features_checked']} checked, "
      f"{dq.summary()['n_failed']} failed -> {dq.failed_by_check()}")

psi_cols = ['feature', 'missing_rate', 'n_unique', 'psi_valid', 'psi_test',
            'psi_max', 'psi_worst_split', 'failed_checks']
dq.report.loc[~dq.report['passed'], psi_cols].round(4)

thresholds applied:
  max_missing_rate   0.95
  min_unique         2
  max_psi            0.25

97 checked, 1 failed -> {'n_unique < 2': 1}


,feature,missing_rate,n_unique,psi_valid,psi_test,psi_max,psi_worst_split,failed_checks
82,net_income_flag,0.0,1,0.0005,0.0005,0.0005,valid,n_unique < 2


In [7]:
# The largest shifts, whether or not they failed. On a random stratified split
# these should all sit far below max_psi.
dq.report.nlargest(5, 'psi_max')[psi_cols].round(4)

,feature,missing_rate,n_unique,psi_valid,psi_test,psi_max,psi_worst_split,failed_checks
17,persistent_eps_in_the_last_four_seasons,0.0,1127,0.0152,0.0207,0.0207,test,
87,cash_total_assets,0.0,4091,0.0191,0.0075,0.0191,valid,
86,cash_reinvestment_pct,0.0,2635,0.0189,0.0083,0.0189,valid,
66,quick_asset_turnover_rate,0.0,3376,0.0182,0.0132,0.0182,valid,
29,current_ratio,0.0,3840,0.0182,0.0077,0.0182,valid,


In [38]:
cfg_strict = load_config('configs/bankruptcy.yaml')
cfg_strict.data_quality.max_psi = 0.015

strict = run_data_quality(dataset, cfg_strict)
print(f'max_psi=0.015 -> {len(strict.failed)} failed  {strict.failed_by_check()}')
strict.report.loc[~strict.report['passed'], ['feature', 'psi_max', 'failed_checks']].round(4).head(6)

max_psi=0.015 -> 15 failed  {'psi_max > 0.015': 14, 'n_unique < 2': 1}


,feature,psi_max,failed_checks
17,persistent_eps_in_the_last_four_seasons,0.0207,psi_max > 0.015
18,revenue_per_share_yuan,0.0171,psi_max > 0.015
19,operating_profit_per_share_yuan,0.0168,psi_max > 0.015
29,current_ratio,0.0182,psi_max > 0.015
38,operating_profit_paid_in_capital,0.0182,psi_max > 0.015
48,operating_profit_per_person,0.0162,psi_max > 0.015


### Stage 2 — feature selection

Two screens on a sample of the training split: **signal** (Spearman rho and mutual information against the outcome) and **redundancy** (the same two measures against the incumbents and against candidates already kept).

In [40]:
selection = run_feature_selection(dataset, cfg)

print(f'kept {len(selection.selected)} of {len(selection.selected) + len(selection.dropped)} candidates')

# One row per candidate. `reached_screen` is how far it got, not what rejected
# it - `kept` and `reason` say that. A candidate cut for having no signal never
# reaches the redundancy screen, so its redundancy columns stay empty.
selection.redundancy_summary[['feature', 'reached_screen', 'max_abs_spearman',
                              'closest_by_spearman', 'kept', 'reason']].round(3)

kept 5 of 13 candidates


,feature,reached_screen,max_abs_spearman,closest_by_spearman,kept,reason
0,cash_turnover_rate,signal,NaN,NaN,False,weak spearman vs target (0.0172)
1,cand_noise,signal,NaN,NaN,False,weak spearman vs target (-0.0066)
2,cand_dup_roa_c,redundancy,0.993,roa_c_before_interest_and_depreciation_before_...,False,redundant with roa_c_before_interest_and_depre...
3,cash_flow_rate,redundancy,0.975,operating_funds_to_liability,False,redundant with operating_funds_to_liability (|...
4,cash_current_liability,redundancy,0.753,quick_ratio,True,
5,cash_flow_per_share,redundancy,0.873,operating_funds_to_liability,True,
6,cfo_to_assets,redundancy,0.954,cash_flow_per_share,False,redundant with cash_flow_per_share (|rho|=0.954)
7,cash_total_assets,redundancy,0.862,cash_current_liability,True,
8,cash_reinvestment_pct,redundancy,0.893,cash_flow_per_share,True,
9,cash_flow_to_sales,redundancy,0.421,cash_total_assets,True,


### Stage 3 — model builds

A *variant* is a named feature list. `leave_one_in` expands to one model per candidate, which is what allows Gini gain to be attributed to a single feature later.

In [41]:
dataset_selected = dataset.with_features(dataset.base_features, selection.selected)

for variant in build_variants(dataset_selected.base_features,
                              dataset_selected.new_features,
                              ['base', 'base_plus_new', 'leave_one_in']):
    print(f'{variant.name:<32} {len(variant.features):>3} features   {variant.note}')

base                              84 features   incumbent features only
base_plus_new                     89 features   incumbent + all selected new
loi__cash_flow_per_share          85 features   base + cash_flow_per_share
loi__cash_reinvestment_pct        85 features   base + cash_reinvestment_pct
loi__cash_total_assets            85 features   base + cash_total_assets
loi__cash_current_liability       85 features   base + cash_current_liability
loi__cash_flow_to_sales           85 features   base + cash_flow_to_sales


In [42]:
cfg_quick = load_config('configs/bankruptcy.yaml')
cfg_quick.run.log_level = 'ERROR'
cfg_quick.model.num_boost_round = 200
# leave_one_in adds one model per surviving candidate: base + that feature and
# nothing else. It is what lets Gini gain be attributed to a single feature.
cfg_quick.model.variants = ['base', 'base_plus_new', 'leave_one_in']
cfg_quick.analysis.include_accuracy = True   # off by default; see section 4.2

models, tuning_log = run_modeling(dataset_selected, cfg_quick,
                                  model_dir=Path(os.environ.get('TMPDIR', '/tmp')) / 'mllite_stage_demo')
for model in models:
    print(f'{model.name:<32} {len(model.features):>3} features   best_iteration={model.best_iteration}')

base                              84 features   best_iteration=199
base_plus_new                     89 features   best_iteration=36
loi__cash_flow_per_share          85 features   best_iteration=196
loi__cash_reinvestment_pct        85 features   best_iteration=187
loi__cash_total_assets            85 features   best_iteration=139
loi__cash_current_liability       85 features   best_iteration=187
loi__cash_flow_to_sales           85 features   best_iteration=190


### Stages 4 and 5 — analysis and verdict

`run_analysis` turns predictions into metrics and SHAP rankings. `run_verdict` is the only stage that reads every other stage's result, because a verdict is exactly the act of combining them.

In [43]:
analysis = run_analysis(dataset_selected, cfg_quick, models)

# Metrics: one row per variant per split.
metric_cols = ['variant', 'split', 'n_rows', 'adj_gini', 'capture_rate_0.05',
               'actual_mean', 'pred_mean']
analysis.metrics[metric_cols].round(4)

,variant,split,n_rows,adj_gini,capture_rate_0.05,actual_mean,pred_mean
0,base,train,4091.0,0.9973,0.9924,0.0323,0.0320
1,base,valid,1364.0,0.9224,0.6591,0.0323,0.0294
2,base,test,1364.0,0.8887,0.5909,0.0323,0.0353
3,base_plus_new,train,4091.0,0.9442,0.7727,0.0323,0.1057
4,base_plus_new,valid,1364.0,0.9128,0.6364,0.0323,0.1059
5,base_plus_new,test,1364.0,0.8941,0.5909,0.0323,0.1108
6,loi__cash_flow_per_share,train,4091.0,0.9966,0.9924,0.0323,0.0321
7,loi__cash_flow_per_share,valid,1364.0,0.9203,0.6136,0.0323,0.0298
8,loi__cash_flow_per_share,test,1364.0,0.8939,0.5682,0.0323,0.0358
9,loi__cash_reinvestment_pct,train,4091.0,0.9963,0.9848,0.0323,0.0321


Each candidate has an *associated variant* — `loi__<feature>`, the incumbent set plus that one feature. Comparing it against `base` prices the feature on its own, which is what the third gate reads:

In [44]:
comparison = analysis.comparison.set_index('variant')

rows = [{'feature': '(baseline)', 'variant': 'base', 'n_features': int(comparison.loc['base', 'n_features'])}]
for feature in dataset_selected.new_features:
    variant = f'loi__{feature}'
    if variant in comparison.index:
        rows.append({'feature': feature, 'variant': variant,
                     'n_features': int(comparison.loc[variant, 'n_features'])})

per_feature = pd.DataFrame(rows)
for column in ['adj_gini_test', 'gini_gain_test', 'accuracy_test',
               'capture_top5_test', 'capture_gain_top5_test']:
    per_feature[column] = [comparison.loc[v, column] for v in per_feature['variant']]

base_accuracy = comparison.loc['base', 'accuracy_test']
per_feature['accuracy_gain_test'] = per_feature['accuracy_test'] - base_accuracy
per_feature = per_feature[["feature", "variant", "n_features", "adj_gini_test", "gini_gain_test", "accuracy_test", "accuracy_gain_test", "capture_top5_test", "capture_gain_top5_test"]]

per_feature.sort_values('gini_gain_test', ascending=False).round(4)

,feature,variant,n_features,adj_gini_test,gini_gain_test,accuracy_test,accuracy_gain_test,capture_top5_test,capture_gain_top5_test
3,cash_total_assets,loi__cash_total_assets,85,0.9030,0.0142,0.7351,0.1194,0.6136,0.0227
4,cash_current_liability,loi__cash_current_liability,85,0.8959,0.0072,0.6476,0.0319,0.5909,0.0000
2,cash_reinvestment_pct,loi__cash_reinvestment_pct,85,0.8946,0.0058,0.6171,0.0014,0.5909,0.0000
1,cash_flow_per_share,loi__cash_flow_per_share,85,0.8939,0.0052,0.5965,-0.0192,0.5682,-0.0227
5,cash_flow_to_sales,loi__cash_flow_to_sales,85,0.8898,0.0010,0.7335,0.1178,0.5682,-0.0227
0,(baseline),base,84,0.8887,0.0000,0.6157,0.0000,0.5909,0.0000


In [45]:
# SHAP: mean |SHAP| per feature, per variant. `shap_share` is that as a
# fraction of the variant's total, so it sums to 1 within each variant.
shap_cols = ['feature', 'mean_abs_shap', 'shap_share', 'shap_rank']
analysis.shap_ranking.query("variant == 'base_plus_new'")[shap_cols].head(10).round(4)

,feature,mean_abs_shap,shap_share,shap_rank
84,net_value_growth_rate,0.0881,0.1538,1
85,borrowing_dependency,0.0812,0.1417,2
86,total_debt_total_net_worth,0.0282,0.0492,3
87,quick_ratio,0.0237,0.0414,4
88,interest_expense_ratio,0.0237,0.0414,5
89,interest_bearing_debt_interest_rate,0.0219,0.0382,6
90,interest_coverage_ratio_interest_expense_to_ebit,0.0209,0.0365,7
91,continuous_interest_rate_after_tax,0.0194,0.0339,8
92,non_industry_income_and_expenditure_revenue,0.0163,0.0284,9
93,net_value_per_share_b,0.0161,0.0281,10


Those are all incumbents. The question this pipeline exists to answer is where the **candidates** land in that same ordering — ranked among every feature of the variant that contains them:

In [46]:
# Rank of each new feature within its own variant's feature set.
# rank_pct = shap_rank / n_features is what verdict.max_shap_rank_pct gates on.
new_features = set(dataset_selected.new_features)

rows = []
for variant, group in analysis.shap_ranking.groupby('variant'):
    if variant.startswith("loi"):
        n_features = len(group)
        for _, row in group[group['feature'].isin(new_features)].iterrows():
            share = row["mean_abs_shap"] / group['mean_abs_shap'].sum()
            equal = 1 / len(group)
            ratio = share / equal
            rows.append({'variant': variant,
                        'feature': row['feature'],
                        'shap_rank': int(row['shap_rank']),
                        'of': n_features,
                        'rank_pct': row['shap_rank'] / n_features,
                        'mean_abs_shap': row['mean_abs_shap'],
                        'shap_share': row['shap_share'],
                        "ratio": ratio})

pd.DataFrame(rows).sort_values(['shap_rank']).round(4)

,variant,feature,shap_rank,of,rank_pct,mean_abs_shap,shap_share,ratio
1,loi__cash_flow_per_share,cash_flow_per_share,9,85,0.1059,0.1572,0.0298,2.5306
2,loi__cash_flow_to_sales,cash_flow_to_sales,9,85,0.1059,0.1706,0.0320,2.7222
4,loi__cash_total_assets,cash_total_assets,18,85,0.2118,0.0810,0.0201,1.7056
3,loi__cash_reinvestment_pct,cash_reinvestment_pct,23,85,0.2706,0.0587,0.0112,0.9559
0,loi__cash_current_liability,cash_current_liability,25,85,0.2941,0.0550,0.0106,0.8968


In [47]:
verdicts, batch = run_verdict(list(dataset.new_features), cfg_quick,
                              dq=dq, fs=selection, analysis=analysis)

print(f'{batch.verdict}  -  {batch.n_passed} of {batch.n_candidates} candidates passed')
print(f'fell at: {batch.failed_at}')

TRY A NEW BATCH  -  0 of 13 candidates passed
fell at: {'feature selection': 8, 'gini gain': 5}


<a id='2'></a>
## 2. One run

`Pipeline` chains the same five calls and writes every artifact to a timestamped directory. This is what the CLI does:

```bash
mllite -c configs/bankruptcy.yaml
mllite -c configs/bankruptcy.yaml --set run.n_jobs=8 --set model.num_boost_round=800
```

In [48]:
result = Pipeline(cfg).run()
print(f'{result.elapsed_seconds:.1f}s  ->  {result.output_dir}')

22.6s  ->  outputs/bankruptcy_cashflow/20260908_151206


In [49]:
for path in sorted(result.output_dir.rglob('*')):
    if path.is_file():
        print(f'{str(path.relative_to(result.output_dir)):<42} {path.stat().st_size / 1024:>8.1f} KB')

batch_verdict.json                              0.5 KB
candidate_verdicts.csv                          3.8 KB
config.resolved.yaml                            2.2 KB
data_quality_report.csv                        11.1 KB
feature_ranking.csv                           214.6 KB
feature_selection_decisions.json                0.8 KB
feature_selection_mi_matrix.csv                27.2 KB
feature_selection_redundancy.csv                1.9 KB
feature_selection_spearman_matrix.csv          27.3 KB
feature_selection_target_stats.csv              1.9 KB
feature_selection_verdicts.csv                  2.5 KB
metrics_by_variant_split.csv                    7.6 KB
models/base.json                              338.1 KB
models/base_plus_new.json                     156.8 KB
models/loi__cand_dup_roa_c.json               299.0 KB
models/loi__cand_noise.json                   287.2 KB
models/loi__cash_current_liability.json       289.8 KB
models/loi__cash_flow_per_share.json          298.5 KB
models/loi

`report.md` is the human-readable summary; `summary.json` the machine-readable one; `config.resolved.yaml` records the exact configuration, so any run can be reproduced from its own output directory.

<a id='3'></a>
### The verdict

Four gates, applied per candidate, then one decision about the batch.

| Gate | Fails when | Threshold |
| --- | --- | --- |
| `data quality` | the column itself is unusable | `data_quality.*` |
| `feature selection` | no signal, or signal the incumbents already carry | `feature_selection.*` |
| `gini gain` | it entered a model and the model did not improve | `verdict.min_gini_gain` |
| `shap rank` | the model kept it but barely uses it | `verdict.max_shap_rank_pct` |

`run.gates` controls whether the gates *act*. This config uses `open`: every candidate is measured at all four gates and nothing is removed, which is the view you want while still learning a dataset. Setting `enforce` makes failing features drop out, and a feature that fails one gate is then never measured at the next.

In [50]:
gate_cols = ['feature', 'verdict', 'failed_at', 'data quality',
             'feature selection', 'gini gain', 'shap rank', 'n_gates_failed']
result.verdicts[gate_cols]

,feature,verdict,failed_at,data quality,feature selection,gini gain,shap rank,n_gates_failed
0,cash_turnover_rate,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,3
1,cash_total_assets,FAIL,gini gain,PASS,PASS,FAIL,PASS,1
2,cash_flow_to_total_assets,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,3
3,cash_flow_to_equity,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,3
4,cand_noise,FAIL,feature selection,PASS,FAIL,FAIL,PASS,2
5,cash_current_liability,FAIL,gini gain,PASS,PASS,FAIL,PASS,1
6,cfo_to_assets,FAIL,feature selection,PASS,FAIL,FAIL,FAIL,3
7,cand_dup_roa_c,FAIL,feature selection,PASS,FAIL,FAIL,PASS,2
8,cash_reinvestment_pct,FAIL,gini gain,PASS,PASS,FAIL,FAIL,2
9,cash_flow_per_share,FAIL,gini gain,PASS,PASS,FAIL,PASS,1


In [51]:
print(json.dumps(result.batch.summary(), indent=2))

{
  "verdict": "TRY A NEW BATCH",
  "n_candidates": 13,
  "n_passed": 0,
  "passed": [],
  "failed_at": {
    "feature selection": 8,
    "gini gain": 5
  },
  "batch_gini_gain": 0.00957,
  "note": "No candidate cleared all four gates - most fell at the feature selection. Propose a different batch rather than relaxing the thresholds. Gates are OPEN (run.gates: open): this is advisory only - nothing was removed, and every candidate was measured at all four gates."
}


No candidate cleared all four gates, so the batch verdict is `TRY A NEW BATCH`.

`failed_at` is the actionable part, because where a batch dies indicates what to try next. Candidates falling at **feature selection** duplicate information the incumbents already carry — look at a different family. Candidates falling at **gini gain** were genuinely new but did not improve the model.

<a id='4'></a>
### The evidence

#### a. Relevance and redundancy

Both directions of mutual information are computed for every candidate, and reported side by side so the trade-off is one sort rather than two thresholds.

| Column | Measures | Want |
| --- | --- | --- |
| `spearman_target` | rank correlation with the outcome | large \|·\| |
| `mi_target` | mutual information with the outcome, in nats | high |
| `nmi_target` | the same, normalized to 0–1 | high |
| `mi_redundancy_base` | normalized MI against the incumbent set | low |
| `mrmr_score` | `nmi_target − mi_redundancy_base` | high |

In [52]:
stat_cols = ['feature', 'spearman_target', 'nmi_target',
             'spearman_redundancy_base', 'mi_redundancy_base', 'mrmr_score']
result.feature_selection.target_stats[stat_cols].round(4)

,feature,spearman_target,nmi_target,spearman_redundancy_base,mi_redundancy_base,mrmr_score
0,cash_flow_rate,-0.1464,0.1144,0.9751,0.5803,-0.4659
1,cash_flow_per_share,-0.1347,0.0919,0.8732,0.3228,-0.2309
2,cash_reinvestment_pct,-0.1014,0.0564,0.8325,0.2631,-0.2067
3,cash_total_assets,-0.1337,0.0907,0.6192,0.1075,-0.0169
4,cash_current_liability,-0.1438,0.1293,0.7525,0.1911,-0.0617
5,cash_turnover_rate,0.0172,0.0381,0.2638,0.1143,-0.0762
6,cash_flow_to_sales,-0.0882,0.0488,0.2888,0.0728,-0.0240
7,cash_flow_to_total_assets,-0.0831,0.0410,0.2995,0.0539,-0.0129
8,cash_flow_to_liability,-0.0748,0.0804,0.2889,0.0960,-0.0157
9,cfo_to_assets,-0.1338,0.0837,0.9472,0.4515,-0.3677


Two details that matter when reading this table.

`nmi_target` adds no ordering information here. The outcome is binary at 3.23%, so `min(H(feature), H(outcome))` is always `H(y)` = 0.1425 nats and `nmi_target` is `mi_target` divided by a constant. It exists so relevance can be subtracted from redundancy, which is also normalized. With a continuous outcome, or with low-cardinality candidates, the two would diverge.

The `_base` columns measure against **incumbents only**, so they are static and order-independent. The redundancy *screen* also compares against candidates already kept, which is why a feature can be dropped for colliding with a fellow candidate while its `spearman_redundancy_base` looks mild.

#### b. Did the model improve?

`gini_gain_<split>` is each variant's adjusted Gini minus the baseline variant's. For a binary outcome the adjusted Gini equals the standard Gini, `2·AUC − 1`.

In [53]:
comparison_cols = ['variant', 'n_features', 'adj_gini_valid', 'adj_gini_test',
                   'gini_gain_valid', 'gini_gain_test', 'capture_top5_test']
result.analysis.comparison[comparison_cols].sort_values(
    'gini_gain_test', ascending=False).round(4)

,variant,n_features,adj_gini_valid,adj_gini_test,gini_gain_valid,gini_gain_test,capture_top5_test
13,loi__cash_turnover_rate,85,0.9072,0.9033,-0.0154,0.0178,0.5909
12,loi__cash_total_assets,85,0.9160,0.9030,-0.0066,0.0175,0.6136
10,loi__cash_flow_to_total_assets,85,0.9036,0.8966,-0.0190,0.0111,0.5909
7,loi__cash_flow_to_equity,85,0.9180,0.8962,-0.0046,0.0108,0.5455
3,loi__cand_noise,85,0.9180,0.8960,-0.0045,0.0105,0.5682
4,loi__cash_current_liability,85,0.9191,0.8959,-0.0035,0.0105,0.5909
14,loi__cfo_to_assets,85,0.9225,0.8956,-0.0001,0.0102,0.5455
1,base_plus_new,97,0.9051,0.8950,-0.0175,0.0096,0.6136
2,loi__cand_dup_roa_c,85,0.9156,0.8949,-0.0070,0.0095,0.5909
11,loi__cash_reinvestment_pct,85,0.9192,0.8946,-0.0034,0.0091,0.5909


The cash-flow family adds no measurable lift: `base_plus_new` sits within a thousandth of `base` on test.

Three limits of this table on a dataset this size, all of which apply to the numbers above:

- **Test holds ~44 bankruptcies.** Gini differences of ±0.01 are inside the noise band.
- **Capture rate is quantised** by the event count: it can only move in steps of 1/44 = 0.023, so a one-step difference between variants is a tie.
- **Read `test` and `valid`, never `train`.** Train Gini is ~0.998 against ~0.89 on test — 95 ratios fitting 4k rows. A gain that appears on train and vanishes on test is the most common false positive this table exists to catch.

#### c. SHAP

`mean_abs_shap` is the average absolute SHAP value per feature, computed on a sample of the test split. `shap_share` is that as a fraction of the variant's total, and `shap_rank` orders features by it. `imp_total_gain_pct` sits beside them for contrast: it is XGBoost's own split-gain importance, summed over every split that used the feature.

In [55]:
ranking = result.analysis.feature_ranking
focus = ranking[ranking['variant'] == 'base_plus_new'].copy()
focus['is_new'] = np.where(focus['feature'].isin(result.dataset.new_features), 'new', '')
focus[['feature', 'is_new', 'mean_abs_shap', 'shap_share', 'shap_rank',
       'imp_total_gain_pct']].head(30).round(4)

,feature,is_new,mean_abs_shap,shap_share,shap_rank,imp_total_gain_pct
84,borrowing_dependency,,0.1543,0.1131,1,0.0740
85,quick_ratio,,0.0995,0.0729,2,0.0685
86,roa_b_before_interest_and_depreciation_after_tax,,0.0922,0.0675,3,0.0484
87,net_value_growth_rate,,0.0899,0.0659,4,0.1051
88,total_debt_total_net_worth,,0.0804,0.0589,5,0.0436
89,persistent_eps_in_the_last_four_seasons,,0.0688,0.0504,6,0.0428
90,interest_bearing_debt_interest_rate,,0.0641,0.0470,7,0.0470
91,interest_expense_ratio,,0.0441,0.0323,8,0.0250
92,debt_ratio_pct,,0.0400,0.0293,9,0.0204
93,continuous_interest_rate_after_tax,,0.0357,0.0262,10,0.0193


The two importance measures disagree, and the disagreement is informative rather than a defect. `total_gain` is measured on the **training** data as the trees were built; SHAP is measured on a **test** sample. A feature can accumulate gain in deep, low-coverage nodes that affect few rows — high total gain, low mean |SHAP|.

In [56]:
# Share of total |SHAP| landing on the new features, against the share they would
# receive from headcount alone. The ratio is the readable number.
rows = []
for variant, group in ranking.groupby('variant'):
    is_new = group['feature'].isin(result.dataset.new_features)
    if not is_new.any():
        continue
    share = group.loc[is_new, 'mean_abs_shap'].sum() / group['mean_abs_shap'].sum()
    equal = is_new.sum() / len(group)
    rows.append({'variant': variant, 'n_new': int(is_new.sum()), 'n_features': len(group), 'shap_share': share, 'equal_split': equal, 'ratio': share / equal})

pd.DataFrame(rows).sort_values('ratio', ascending=False).round(3).head(8)

,variant,n_new,n_features,shap_share,equal_split,ratio
4,loi__cash_flow_per_share,1,85,0.030,0.012,2.531
8,loi__cash_flow_to_sales,1,85,0.029,0.012,2.484
6,loi__cash_flow_to_equity,1,85,0.026,0.012,2.209
7,loi__cash_flow_to_liability,1,85,0.025,0.012,2.118
13,loi__cfo_to_assets,1,85,0.023,0.012,1.936
5,loi__cash_flow_rate,1,85,0.022,0.012,1.887
11,loi__cash_total_assets,1,85,0.020,0.012,1.706
9,loi__cash_flow_to_total_assets,1,85,0.017,0.012,1.403


A share is only meaningful against that benchmark. If every feature contributed equally, `n_new / n_features` would go to the candidates by arithmetic alone, so a ratio near 1.0 means the candidates are pulling exactly their headcount weight and nothing more.

**Attribution is not value.** SHAP measures what the model *uses*; Gini gain measures what it *gained*. When a candidate correlates with incumbents the model substitutes it in, credit moves onto it, and the ranking does not improve. This is why the verdict gates on Gini and uses SHAP rank only as a final low-usage filter.

<a id='6'></a>
## 3. The discovery loop

One run answers one question, and the answer determines what to propose next. `verify()` wraps a run into a single call: a frame and a candidate list in, a verdict out. Candidates are engineered in memory and passed straight through, so nothing round-trips through a file between rounds.

In [24]:
def verify(frame, candidates, *, name, min_gini_gain=0.005, boost_rounds=250):
    """Run the protocol over one candidate batch. Returns (summary dict, PipelineResult)."""
    cfg = load_config('configs/bankruptcy.yaml')
    cfg.run.name = name
    cfg.run.log_level = 'ERROR'
    cfg.features.new = list(candidates)
    cfg.model.num_boost_round = boost_rounds
    # leave_one_in gives each candidate its own model, which is what lets the
    # gini gate be attributed per feature rather than only to the batch.
    cfg.model.variants = ['base', 'base_plus_new', 'leave_one_in']
    cfg.verdict.min_gini_gain = min_gini_gain

    run = Pipeline(cfg).run(frame=frame)
    comparison = run.analysis.comparison.set_index('variant')
    summary = {
        'round': name,
        'proposed': len(candidates),
        'survived_screens': len(run.feature_selection.selected),
        'gini_base': float(comparison.loc['base', 'adj_gini_test']),
        'gini_with_new': float(comparison.loc['base_plus_new', 'adj_gini_test']),
        'gain_valid': float(comparison.loc['base_plus_new', 'gini_gain_valid']),
        'gain_test': float(comparison.loc['base_plus_new', 'gini_gain_test']),
        'verdict': run.batch.verdict,
        'fell_at': dict(run.batch.failed_at),
    }
    return summary, run

### Round 1 — the cash-flow family

In [25]:
CASH_FLOW = [c for c in frame.columns if c.startswith('cash') or c == 'cfo_to_assets']
round_1, run_1 = verify(frame, CASH_FLOW, name='loop_r1_cashflow')
round_1

{'round': 'loop_r1_cashflow',
 'proposed': 11,
 'survived_screens': 5,
 'gini_base': 0.8935950413223142,
 'gini_with_new': 0.9022382920110193,
 'gain_valid': -0.018767217630854094,
 'gain_test': 0.008643250688705195,
 'verdict': 'TRY A NEW BATCH',
 'fell_at': {'feature selection': 6, 'gini gain': 5}}

### Round 2 — interactions between the model's strongest drivers

A negative result indicates where to look next. Round 1 showed the cash-flow ratios are already spanned by the incumbents, so proposing more standalone ratios is unlikely to help. Interactions between the features SHAP ranks highest are a different signal shape.

In [26]:
top_drivers = (result.analysis.feature_ranking
               .query("variant == 'base_plus_new'")
               .head(6)['feature'].tolist())

engineered = {}
for i, a in enumerate(top_drivers):
    for b in top_drivers[i + 1:]:
        engineered[f'x__{a[:14]}__{b[:14]}'] = frame[a] * frame[b]
        engineered[f'r__{a[:14]}__{b[:14]}'] = frame[a] / frame[b].replace(0, np.nan)

round_2, run_2 = verify(frame.assign(**engineered), list(engineered),
                        name='loop_r2_interactions')
round_2

{'round': 'loop_r2_interactions',
 'proposed': 30,
 'survived_screens': 0,
 'gini_base': 0.8977617079889807,
 'gini_with_new': 0.9066460055096416,
 'gain_valid': 0.012086776859504544,
 'gain_test': 0.008884297520660889,
 'verdict': 'TRY A NEW BATCH',
 'fell_at': {'feature selection': 30}}

In [27]:
# Each surviving candidate also gets a base + one-feature model, which is what the
# per-feature gini gate reads.
loi = run_2.analysis.comparison
loi = loi[loi['variant'].str.startswith('loi__')]
loi[['variant', 'gini_gain_valid', 'gini_gain_test']].sort_values(
    'gini_gain_test', ascending=False).head(6).round(4)

,variant,gini_gain_valid,gini_gain_test
8,loi__r__net_value_grow__total_debt_tot,-0.0023,0.0048
25,loi__x__quick_ratio__persistent_eps,0.0010,0.0043
24,loi__x__quick_ratio__net_value_grow,-0.0039,-0.0002
19,loi__x__borrowing_depe__quick_ratio,-0.0019,-0.0007
3,loi__r__borrowing_depe__persistent_eps,-0.0003,-0.0019
28,loi__x__roa_b_before_i__net_value_grow,-0.0021,-0.0020


In [28]:
pd.DataFrame([round_1, round_2]).round(4)

,round,proposed,survived_screens,gini_base,gini_with_new,gain_valid,gain_test,verdict,fell_at
0,loop_r1_cashflow,11,5,0.8936,0.9022,-0.0188,0.0086,TRY A NEW BATCH,"{'feature selection': 6, 'gini gain': 5}"
1,loop_r2_interactions,30,0,0.8978,0.9066,0.0121,0.0089,TRY A NEW BATCH,{'feature selection': 30}


Both batches fail, and round 2 is the more instructive of the two: the interactions absorb a large share of SHAP attribution while gaining nothing on held-out data. A gradient-boosted ensemble already composes interactions from the raw ratios, so supplying them pre-multiplied moves where credit lands without changing how well the model ranks.

Two batches, two failures, and that is a result rather than a dead end: on this dataset the 84 incumbent ratios look saturated, and the next move is new *data* rather than further arithmetic on the same columns.

**Two limits of the protocol, worth stating explicitly.** The thresholds live in the config (`verdict.min_gini_gain`, `verdict.require_valid_too`, `verdict.max_shap_rank_pct`) rather than in this notebook, so they are reviewable in one place — but they remain blunt: with ~44 events in test, a 0.005 gain threshold sits inside the noise band, so bootstrap the gain or repeat across seeds before acting on a `KEEP`. And nothing here detects leakage: a feature built from the outcome clears all four gates and posts a large gain.

<a id='7'></a>
## Configuration

### Parallelism

`run.n_jobs` and `run.backend` govern every stage: feature-selection chunks, one task per model variant, one per variant for metrics and SHAP. XGBoost brings its own thread pool, so the two layers are budgeted against each other rather than oversubscribing the machine. Setting `run.backend: sequential` produces identical results, which the test suite asserts.

In [29]:
from mllite.parallel import resolve_n_jobs, threads_per_worker

for n_jobs in (-1, 4, 1):
    print(f'n_jobs={n_jobs:>2} -> {resolve_n_jobs(n_jobs)} worker(s), '
          f'{threads_per_worker(n_jobs, n_tasks=8)} xgboost thread(s) each')

n_jobs=-1 -> 8 worker(s), 1 xgboost thread(s) each
n_jobs= 4 -> 4 worker(s), 2 xgboost thread(s) each
n_jobs= 1 -> 1 worker(s), 8 xgboost thread(s) each


### Hyperparameters

`model.tuning` runs a random search scored on the valid split. `mode` is the choice that matters for a champion/challenger comparison:

- `shared` — tune once on `tune_on`, then give every variant the same hyperparameters, so the Gini differences are attributable to the feature sets.
- `per_variant` — tune each variant separately. Fairer to a challenger whose optimum genuinely differs, at the cost of mixing tuning variance into the gap.

On this dataset the same candidates measured +0.0034 under `shared` and +0.0065 under `per_variant`, which straddles the default 0.005 gate. Prefer `shared` for verdict runs.

### Strictness

Unknown keys raise at load time rather than being silently ignored, so a mistyped threshold fails immediately instead of quietly changing what a run means.

In [30]:
try:
    load_config('configs/bankruptcy.yaml',
                {'feature_selection.spearman.target_min': 0.1})
except ValueError as exc:
    print('caught:', exc)

caught: SpearmanConfig: unknown config key(s): ['target_min']
